1) Campos de Galois:

    Diseñar una clase en Python que represente un campo de Galois GF(2^m).

    - El constructor debe recibir el orden m del campo y el polinomio primitivo P(x), representado como un entero de m bits (sin el termino x^m, que queda implícito en la reducción)

    - Los elementos del campo se representan como enteros en el rango [0, 2^m-1]

    - La clase debe implementar las siguientes operaciones sobre elementos del campo:

            - Suma
            - Producto
            - Inverso multiplicativo (indicando el comportamiento para el elemento 0)
            - Division (indicando el comportamiento para el divisor 0)
            - Potencia A^n, para n>=0

    - Los elementos del campo se representan como enteros en el rango [0,2^m-1], el resultado de interpretar la tupla de m coeficientes binarios como un número en base 2


In [1]:
class GF:
    def __init__(self, m: int, p_x: int):
        
        self.m = m   #orden del campo
        self.p_x = p_x  #polinomio primitivo representado como entero de m bits
        self.order = 2** m  # 2^m
        self.mask = self.order - 1  #2^m-1

    def validar_elemento(self, elem: int):
        if not (0 <= elem < self.order):
            raise ValueError(f"El elemento {elem} no pertenece a GF(2^{self.m})")
        #else:
            #print(f"{elem} es elemento del cuerpo de Galois 2^{self.m}")

    #Suma (y resta) en GF(2^m): operación XOR bit a bit.
    def sum(self, a: int, b: int) -> int:
        self.validar_elemento(a)
        self.validar_elemento(b)
        return a ^ b

    #Multiplicación en GF(2^m) usando desplazamientos y reducción por el polinomio primitivo.
    def mul(self, a: int, b: int) -> int:
        self.validar_elemento(a)
        self.validar_elemento(b)

        result = 0
        
        for _ in range(self.m):
            if (b & 1):
                result ^= a
            
            # Verifica si el bit m-1 de 'a' está activo antes de desplazar
            carry = a & (1 << (self.m - 1))
            a <<= 1  

            if carry:
                # Se aplica XOR con el polinomio primitivo completo (x^m + p_x)
                a ^= self.p_x

            b >>= 1

        return result & self.mask

#Potencia 
    def pot(self, base: int, exp: int) -> int:
        self.validar_elemento(base)
        if exp<0:
            raise ValueError("El exponente debe ser mayor o igual a 0")
        res = 1
        for _ in range(exp):
            res = self.mul(res, base)
        return res

#Inverso multiplicativo 
#a^(-1) = a^(2^m - 2) en GF(2^m).

    def inv(self, a: int) -> int:
        self.validar_elemento(a)
        if a == 0:
            raise ZeroDivisionError("El elemento 0 no tiene inverso multiplicativo en GF(2^m).")
        return self.pot(a, self.order - 2)

#División a/b
    def div(self, a: int, b: int) -> int:
        self.validar_elemento(a)
        self.validar_elemento(b)
        if b == 0:
            raise ZeroDivisionError("División por cero no permitida en GF(2^m).")
        return self.mul(a, self.inv(b))


In [2]:
#Verificación de la clase

#Polinomio primitivo: P(x) = X^4+X+1 -> bin: 1|0011 -> dec=3 (sin el término X^m)
gf4=GF(m=4,p_x=3) 
print (f"Cuerpo de Galois (2^{gf4.m})")
print (f"Polinomio primitivo p(X)={gf4.p_x} (binario:{bin(gf4.p_x)})")
print(f"Elementos van de 0 a {gf4.mask}\n")

#Polinomio primitivo: P(x)=X^4+X^3+1 -> bin:1|1001 -> dec=9 (sin el término X^m)
gf4_1=GF(m=4,p_x=9) 
print (f"Cuerpo de Galois (2^{gf4_1.m})")
print (f"Polinomio primitivo p(X)={gf4_1.p_x} (binario:{bin(gf4_1.p_x)})")
print(f"Elementos van de 0 a {gf4_1.mask}")

Cuerpo de Galois (2^4)
Polinomio primitivo p(X)=3 (binario:0b11)
Elementos van de 0 a 15

Cuerpo de Galois (2^4)
Polinomio primitivo p(X)=9 (binario:0b1001)
Elementos van de 0 a 15


In [3]:
#Validación de elementos
a=1
b=5
c=17

gf4.validar_elemento(a)
gf4.validar_elemento(b)
#gf4.validar_elemento(c)


In [4]:
#Suma y Multiplicación

#Suma
resultado=gf4.sum(a,b)
print(f"Decimal: {a}+{b}= {resultado}")
print(f"Binario: {bin(a)[2:].zfill(4)} + {bin(b)[2:].zfill(4)} = {bin(resultado)[2:].zfill(4)}\n")

# Multiplicación
resultado1=gf4.mul(a,b)
print(f"Decimal: {a}*{b}= {resultado1}")
print(f"Binario: {bin(a)[2:].zfill(4)} * {bin(b)[2:].zfill(4)} = {bin(resultado1)[2:].zfill(4)}")


Decimal: 1+5= 4
Binario: 0001 + 0101 = 0100

Decimal: 1*5= 5
Binario: 0001 * 0101 = 0101


In [5]:
#Inverso multiplicativo

resultado2=gf4.inv(a)
print(f"Decimal: {a}^(-1)= {resultado2}")
print(f"Binario: {bin(a)[2:].zfill(4)}^(-1) = {bin(resultado2)[2:].zfill(4)}\n")

resultado3=gf4.inv(b)
print(f"Decimal: {b}^(-1)= {resultado3}")
print(f"Binario: {bin(b)[2:].zfill(4)}^(-1) = {bin(resultado3)[2:].zfill(4)}")

Decimal: 1^(-1)= 1
Binario: 0001^(-1) = 0001

Decimal: 5^(-1)= 11
Binario: 0101^(-1) = 1011


In [6]:
#División 

resultado4=gf4.div(a,b)
print(f"Decimal: {a}/{b}= {resultado4}")
print(f"Binario: {bin(a)[2:].zfill(4)} / {bin(b)[2:].zfill(4)} = {bin(resultado4)[2:].zfill(4)}")

Decimal: 1/5= 11
Binario: 0001 / 0101 = 1011


In [7]:
#Potencia
n=2
resultado5=gf4.pot(a,n)
print(f"Decimal: {a}^({n})= {resultado5}")
print(f"Binario: {bin(a)[2:].zfill(4)}^({n}) = {bin(resultado5)[2:].zfill(4)}")

resultado6=gf4.pot(b,n)
print(f"Decimal: {b}^({n})= {resultado6}")
print(f"Binario: {bin(b)[2:].zfill(4)}^({n}) = {bin(resultado6)[2:].zfill(4)}")

Decimal: 1^(2)= 1
Binario: 0001^(2) = 0001
Decimal: 5^(2)= 2
Binario: 0101^(2) = 0010


2) Polinomio sobre un Campo de Galois
    Utilizando la clase desarrollada previamente, diseñar una clase GFPoly que represente un polinomio con coeficientes en GF(2^m).
    - El constructor debe recibir una instancia de la clase anterior y una lista (o tupla) de coeficientes en orden decreciente de grado.
    - Debe validarse que cada coeficiente sea un elemento valido del campo y que la representación interna no contenga ceros a la izquierda salvo en el caso del polinomio nulo.
    - La clase debe implementar las siguientes operaciones, todas ellas realizadas sobre el campo GF(2^m) subyacente:

        - Suma de dos polinomios
        - Producto de dos polinomios.
        - División entera de dos polinomios, obteniendo por separado el cociente y el resto.
        - Escalado: multiplicar todos los coeficientes de un polinomio por un escalar del campo.
        - Evaluación del polinomio en un punto x ∈ GF(2m).
        - Construcción a partir de raíces: dado un conjunto de raíces {r1,...,rk} ⊂ GF(2m), construir el polinomio ki=1(x − ri).
        
    - Se recomienda sobrecargar los operadores de Python correspondientes (+, *, //,%, ==) para que las operaciones puedan escribirse de forma natural sobre objetos de la clase.

In [8]:

class GFPoly:
    def __init__(self, gf: GF, coeffs: list[int]):
        
        #Representa un polinomio con coeficientes en GF(2^m).
        # Parámetro gf: Instancia de la clase GF.
        # Parámetro coeffs: Lista de coeficientes en orden decreciente de grado [a_n, ..., a_0].
        
        self.gf = gf

        # Validar que cada elemento sea un elemento válido del campo GF(2^m)
        for c in coeffs:
            self.gf.validar_elemento(c)

        # Eliminar ceros a la izquierda 
        i = 0
        while i < len(coeffs) - 1 and coeffs[i] == 0:
            i += 1
        
        self.coeffs = list(coeffs[i:]) if coeffs else [0]

    @property
    def grado(self) -> int:
        if self.coeffs == [0]:
            return -1
        return len(self.coeffs) - 1

    def validar_campos (self, other: "GFPoly") -> "GFPoly":
        if self.gf != other.gf:
            raise ValueError("Los polinomios deben estar sobre el mismo campo GF.")


    #SUMA
    def suma(self, other: "GFPoly") -> "GFPoly":
        self.validar_campos (other)

        len1, len2 = len(self.coeffs), len(other.coeffs)
        max_len = max(len1, len2)

        # Alinear coeficientes agregando ceros a la izquierda
        p1 = [0] * (max_len - len1) + self.coeffs
        p2 = [0] * (max_len - len2) + other.coeffs

        res_coeffs = [self.gf.sum(a, b) for a, b in zip(p1, p2)]
        return GFPoly(self.gf, res_coeffs)


    #PRODUCTO
    def producto(self, other: "GFPoly") -> "GFPoly":
        self.validar_campos (other)

        if self.grado == -1 or other.grado == -1:
            return GFPoly(self.gf, [0])

        res_len = len(self.coeffs) + len(other.coeffs) - 1
        res = [0] * res_len

        for i, c1 in enumerate(self.coeffs):
            for j, c2 in enumerate(other.coeffs):
                prod = self.gf.mul(c1, c2)
                res[i + j] = self.gf.sum(res[i + j], prod)

        return GFPoly(self.gf, res)


    #DIVISION entera de dos polinomios, obteniendo por separado el cociente y el resto
    def divmod(self, other: "GFPoly") -> tuple["GFPoly", "GFPoly"]:
        self.validar_campos (other)

        if other.grado == -1:
            raise ZeroDivisionError("División entera por el polinomio nulo.")

        if self.grado < other.grado:
            print(f"El grado del polinomio F(X)=({other}) es {other.grado}")
            print(f"menor que el grado del divisor G(X)={self} el cual es {self.grado}")
            return GFPoly(self.gf, [0]), GFPoly(self.gf, self.coeffs)

        rem = list(self.coeffs)
        deg_divisor = other.grado #Grado del divisor
        lead_b = other.coeffs[0]

        cociente = [0] * (len(rem) - deg_divisor)

        while len(rem) - 1 >= deg_divisor and rem != [0]:
            shift = len(rem) - 1 - deg_divisor
            coeff = self.gf.div(rem[0], lead_b)
            cociente[len(cociente) - 1 - shift] = coeff

            # Restar (sumar) coeff * other * x^shift
            for i, c in enumerate(other.coeffs):
                prod = self.gf.mul(coeff, c)
                rem[i] = self.gf.sum(rem[i], prod)

            # Quitar ceros principales
            while len(rem) > 1 and rem[0] == 0:
                rem.pop(0)

        return GFPoly(self.gf, cociente), GFPoly(self.gf, rem)

    

    def __floordiv__(self, other: "GFPoly") -> "GFPoly":
        return self.divmod(other)[0]
    

    #Resto
    def __mod__(self, other: "GFPoly") -> "GFPoly":
        return self.divmod(other)[1]

    
    #ESCALADO: multiplicar todos los coeficientes de un polinomio por un escalar del campo
    def scale(self, scalar: int) -> "GFPoly":
        self.gf.validar_elemento(scalar)
        if scalar == 0:
            return GFPoly(self.gf, [0])
        return GFPoly(self.gf, [self.gf.mul(c, scalar) for c in self.coeffs])


    #EVALUACIÓN del polinomio en un punto x perteneciente a GF(2^m)
    def eval(self, x: int) -> int:
        self.gf.validar_elemento(x)
        res = 0
        for c in self.coeffs:
            res = self.gf.sum(self.gf.mul(res, x), c)
        return res


    #Raíces
    @classmethod
    def desde_raices(cls, gf: GF, raices: list[int]) -> "GFPoly":

        #Construye el polinomio P(x)=(x - r1)(x - r2)... a partir de una lista de raíces.

        poly = cls(gf, [1])  # Polinomio unitario P(x) = 1

        for r in raices:
            gf.validar_elemento(r)
            term = cls(gf, [1, r]) #Crea el factor (X+r)
            poly = poly * term
        return poly


    #Igualdad
    def __eq__(self, other: object) -> bool:
        if not isinstance(other, GFPoly):
            return False
        return self.gf == other.gf and self.coeffs == other.coeffs


    #Representación en forma de Polinomio
    def __repr__(self) -> str:
        if self.coeffs == [0]:
            return "0"
        deg = len(self.coeffs) - 1
        terms = []
        for i, c in enumerate(self.coeffs):
            power = deg - i
            if c != 0:
                if power == 0:
                    terms.append(f"{c}")
                elif power == 1:
                    terms.append(f"{c}x" if c != 1 else "x")
                else:
                    terms.append(f"{c}x^{power}" if c != 1 else f"x^{power}")
        return " + ".join(terms)

In [9]:
P=[3,5,1]  
F=[7,0,4]   
G=[3,2,5,1]
H=[4,7]

poly1=GFPoly(gf4,P)
poly2=GFPoly(gf4,F)
poly3=GFPoly(gf4,G)
poly4=GFPoly(gf4,H)


In [10]:
#Suma
resul=poly1.suma(poly2)
print(f"Suma: F(X)={poly1} + G(X)= {poly2 }")
print(f"\tes igual a  {resul}\n")


#Multiplicación
resul1=poly1.producto(poly2)
print(f"Producto: F(X)={poly1} * G(X)= {poly2 } ")
print(f"\tes igual a  {resul1}")

Suma: F(X)=3x^2 + 5x + 1 + G(X)= 7x^2 + 4
	es igual a  4x^2 + 5x + 5

Producto: F(X)=3x^2 + 5x + 1 * G(X)= 7x^2 + 4 
	es igual a  9x^4 + 8x^3 + 11x^2 + 7x + 4


In [11]:
#Division
cociente , resto =poly3.divmod(poly4)
print(f"Divido F(X)= {poly3} en G(X)= {poly4}")
print(f"Cociente: Q(X)= {cociente}")
print(f"Resto: R(X)= {resto}\n")

print(f"Divido F(X)= {poly4} en G(X)= {poly3}")
cociente1 , resto1 =poly4.divmod(poly3)
print(f"Cociente: {cociente1}")
print(f"Resto: {resto1}")


Divido F(X)= 3x^3 + 2x^2 + 5x + 1 en G(X)= 4x + 7
Cociente: Q(X)= 4x^2 + 14x + 15
Resto: R(X)= 10

Divido F(X)= 4x + 7 en G(X)= 3x^3 + 2x^2 + 5x + 1
El grado del polinomio F(X)=(3x^3 + 2x^2 + 5x + 1) es 3
menor que el grado del divisor G(X)=4x + 7 el cual es 1
Cociente: 0
Resto: 4x + 7


In [12]:
#Escalar:
y=3
x=1

result5=poly4.scale(y)
print(f"Producto Escalar Q(x)= {y}* [P(x)={poly4}] = {result5}\n")

#Evaluación
result6=poly4.eval(x)
print(f"F(X)= {poly4} particularizo para X={x}")
print(f"F({x})={result6}")

Producto Escalar Q(x)= 3* [P(x)=4x + 7] = 12x + 9

F(X)= 4x + 7 particularizo para X=1
F(1)=3
